# TrustShift-IDS — WP2 Version 9: neural detector-specific primary monitor

**Goal:** replace the V5 median-displacement proxy with the original V2.8C-R10 12-indicator, three-family monitoring structure. Each neural detector uses its own **binary and multiclass** models. **No model retraining.**

**Methodological status:** This is a *new detector-specific extension*, not an exact replay of the historical XGBoost monitor. Its normalization and state boundaries are estimated using **source-only calibration windows** and frozen before deployment evaluation. It does not borrow XGBoost state thresholds or use deployment labels to calibrate the monitor.

**Safeguards:** all monitoring uses unlabeled inputs/predictions; deployment labels are loaded only in the outcome cell; a missing historical transferred-state baseline is reported as unavailable, never synthesized. Save intermediate artifacts to Drive. GPU recommended for neural inference, not necessary for Wasserstein.

**Potential limitations:** calibration windows overlap the full source reference used for deployment (while each null comparison uses a leave-window-out reference for predictive evidence); source-null finite-sample quantiles can be noisy with 54 windows. Threshold selection for abstention follows the V4 protocol and is reported explicitly rather than described as a formal risk guarantee.

In [ ]:
%pip -q install "numpy<2.1" pandas scipy pyarrow joblib tensorflow scikit-learn
from google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')


In [ ]:
# CELL N0 — setup, custom layers, model loading, reproduction gates
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
import os, gc, json, glob, time, shutil
import numpy as np, pandas as pd, tensorflow as tf, keras
BASE = "/content/drive/MyDrive/TrustShift_IDS"
V2 = f"{BASE}/V2_adaptive_reliability"
OUT = f"{V2}/revision_R4"; os.makedirs(OUT, exist_ok=True)
WINDOW, EPS = 5000, 0.01
print("TF", tf.__version__, "| Keras", keras.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

# --- custom layers, verbatim from V1 Step 6.3D (needed to deserialise the FT-Transformer) ---
@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, n_features, token_dim, **kwargs):
        super().__init__(**kwargs); self.n_features = int(n_features); self.token_dim = int(token_dim)
    def build(self, input_shape):
        self.feature_weight = self.add_weight(name="feature_weight", shape=(self.n_features, self.token_dim), initializer="glorot_uniform", trainable=True)
        self.feature_bias = self.add_weight(name="feature_bias", shape=(self.n_features, self.token_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.expand_dims(inputs, axis=-1) * self.feature_weight + self.feature_bias
    def get_config(self):
        c = super().get_config(); c.update({"n_features": self.n_features, "token_dim": self.token_dim}); return c

@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class CLSToken(tf.keras.layers.Layer):
    def __init__(self, token_dim, **kwargs):
        super().__init__(**kwargs); self.token_dim = int(token_dim)
    def build(self, input_shape):
        self.cls_token = self.add_weight(name="cls_token", shape=(1, 1, self.token_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.concat([tf.tile(self.cls_token, [tf.shape(inputs)[0], 1, 1]), inputs], axis=1)
    def get_config(self):
        c = super().get_config(); c.update({"token_dim": self.token_dim}); return c

CUSTOM = {"NumericalFeatureTokenizer": NumericalFeatureTokenizer, "CLSToken": CLSToken,
          "TrustShiftIDS>NumericalFeatureTokenizer": NumericalFeatureTokenizer, "TrustShiftIDS>CLSToken": CLSToken}
MODELS = {"MLP": f"{BASE}/models/mlp/MLP_binary_seed42.keras",
          "FTTransformer": f"{BASE}/models/ft_transformer/FTTransformer_binary_seed42.keras"}
M = {}
for name, p in MODELS.items():
    M[name] = keras.models.load_model(p, compile=False, custom_objects=CUSTOM, safe_mode=False)
    print(f"[N0] loaded {name}: input {M[name].input_shape} output {M[name].output_shape}")

# --- frozen neural preprocessing (V1 Step 5.5C): clip + standardise continuous features only ---
SPEC = pd.read_csv(f"{BASE}/metadata/TrustShift_IDS_FROZEN_NEURAL_PREPROCESSING_SPEC.csv")
assert len(SPEC) == 68
FEATURES = SPEC["feature_id"].tolist()
CONT = SPEC[SPEC.feature_group == "continuous"]
assert len(CONT) == 56
CONT_IDX = np.array([FEATURES.index(f) for f in CONT.feature_id]); LO = CONT.lower_bound.to_numpy(float); HI = CONT.upper_bound.to_numpy(float)
MU = CONT["mean"].to_numpy(float); SD = CONT["std"].to_numpy(float)
def neural_transform(X):
    X = np.array(X, dtype=np.float64, copy=True)
    X[:, CONT_IDX] = (np.clip(X[:, CONT_IDX], LO, HI) - MU) / SD
    return X

def p_attack(model, Xn, bs=16384):
    out = model.predict(Xn.astype(np.float32), batch_size=bs, verbose=0)
    return np.asarray(out, dtype=np.float64).reshape(-1)

# Gate G1: transform(IID imputed) must equal stored IID neural representation
P_IMP = f"{BASE}/data/processed/CICIDS2017_68F_imputed"; P_NEU = f"{BASE}/data/processed/CICIDS2017_68F_neural"
iid_imp = pd.read_parquet(f"{P_IMP}/CICIDS2017_iid_test_68F_imputed.parquet")
iid_neu = pd.read_parquet(f"{P_NEU}/CICIDS2017_iid_test_68F_neural.parquet", columns=FEATURES)
d = float(np.max(np.abs(neural_transform(iid_imp[FEATURES].to_numpy()) - iid_neu.to_numpy(np.float64))))
print(f"[N0] G1 transform reproduction: max|diff| = {d:.3e}", "PASS" if d < 1e-9 else "FAIL"); assert d < 1e-9
raw = iid_imp["binary_label"]
y_iid = raw.to_numpy(np.int64) if np.issubdtype(raw.dtype, np.number) else raw.astype(str).str.lower().map({"benign": 0, "attack": 1, "0": 0, "1": 1}).to_numpy(np.int64)
X_iid_raw = iid_imp[FEATURES].to_numpy(np.float64); del iid_imp, iid_neu; gc.collect()

# Gate G2: reproduce the saved V1 validation predictions
val_neu = pd.read_parquet(f"{P_NEU}/CICIDS2017_validation_68F_neural.parquet", columns=FEATURES).to_numpy(np.float64)
for name in M:
    sp = f"{BASE}/results/{'mlp' if name=='MLP' else 'ft_transformer'}/{name}_binary_seed42_validation_predictions.parquet"
    saved = pd.read_parquet(sp); print(f"[N0] saved {name} validation columns: {list(saved.columns)}")
    cand = [c for c in saved.columns if any(k in c.lower() for k in ["prob", "p1", "p_attack", "score", "proba"])]
    p_now = p_attack(M[name], val_neu)
    if cand:
        dd = float(np.max(np.abs(p_now - saved[cand[0]].to_numpy(np.float64))))
        print(f"[N0] G2 {name}: column '{cand[0]}' max|diff| = {dd:.3e}", "PASS" if dd < 1e-4 else "CHECK")
    else:
        print(f"[N0] G2 {name}: no probability column recognised — paste the column list above")
del val_neu; gc.collect()
print("[N0] COMPLETE")

In [ ]:
# CELL N1 — temperature scaling on calibration, thresholds, ATC, IID test, 54 source-null windows
from scipy.optimize import minimize_scalar
cal = pd.read_parquet(f"{P_NEU}/CICIDS2017_calibration_68F_neural.parquet")
assert len(cal) == 274787
Xc = cal[FEATURES].to_numpy(np.float64)
rc = cal["binary_label"]; yc = rc.to_numpy(np.int64) if np.issubdtype(rc.dtype, np.number) else rc.astype(str).str.lower().map({"benign":0,"attack":1,"0":0,"1":1}).to_numpy(np.int64)
del cal; gc.collect()
COV_GRID = [0.999, 0.995, 0.99, 0.98, 0.975, 0.95, 0.925, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
def logit(p): p = np.clip(p, 1e-12, 1 - 1e-12); return np.log(p) - np.log1p(-p)
def calibrated(p, T): return 1.0 / (1.0 + np.exp(-logit(p) / T))
def risk_pred(p, T):
    q = calibrated(p, T); pred = (q >= 0.5).astype(np.int64); return 1.0 - np.maximum(q, 1 - q), pred
CAL = {}
for name, model in M.items():
    p = p_attack(model, Xc)
    nll = lambda logT: float(np.mean(-(yc * np.log(np.clip(calibrated(p, np.exp(logT)), 1e-15, 1)) + (1 - yc) * np.log(np.clip(1 - calibrated(p, np.exp(logT)), 1e-15, 1)))))
    T = float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))
    r, pr = risk_pred(p, T); err = pr != yc
    # fixed threshold: loosest threshold whose calibration selective risk <= 1%
    order = np.argsort(r, kind="mergesort"); cum_err = np.cumsum(err[order]); k = np.arange(1, len(r) + 1)
    ok = np.where(cum_err / k <= EPS)[0]; tau_fixed = float(r[order][ok.max()])
    thr = {"fixed": tau_fixed}
    rs = np.sort(r)
    for c in [0.99, 0.90]: thr[{0.99: "elevated", 0.90: "severe"}[c]] = float(rs[int(np.ceil(c * len(r))) - 1])
    for c in COV_GRID: thr[f"cov_{c:.3f}"] = float(rs[int(np.ceil(c * len(r))) - 1])
    conf = 1 - r; e_src = float(err.mean()); atc_t = float(np.quantile(conf, e_src))
    perm = np.random.default_rng(42).permutation(len(r))[:54 * WINDOW].reshape(54, WINDOW)
    null_atc = []
    for w in range(54):
        idx = perm[w]; rest = np.ones(len(r), bool); rest[idx] = False
        null_atc.append(float(np.mean(conf[idx] < np.quantile(conf[rest], float(err[rest].mean())))))
    atc_b = [float(np.quantile(null_atc, 0.95)), float(np.quantile(null_atc, 0.99))]
    ri, pi = risk_pred(p_attack(model, neural_transform(X_iid_raw)), T); ei = pi != y_iid
    CAL[name] = {"T": T, "thresholds": thr, "atc_conf_threshold": atc_t, "atc_boundaries": atc_b, "cal_errors": int(err.sum()),
                 "cal_cov_fixed": float(np.mean(r <= tau_fixed)), "iid_errors": int(ei.sum()), "iid_accuracy": float(1 - ei.mean()),
                 "iid_fixed_coverage": float(np.mean(ri <= tau_fixed)),
                 "iid_fixed_risk": float(ei[ri <= tau_fixed].mean()) if (ri <= tau_fixed).any() else None}
    print(f"[N1] {name}: T={T:.4f} | cal errors {err.sum()} | fixed tau {tau_fixed:.4g} (cal cov {CAL[name]['cal_cov_fixed']:.4f}) | "
          f"IID errors {ei.sum()} acc {CAL[name]['iid_accuracy']:.5f} | ATC bounds {atc_b}")
json.dump(CAL, open(f"{OUT}/N1_neural_calibration_and_thresholds.json", "w"), indent=2)
del Xc; gc.collect(); print("[N1] COMPLETE")

## N2 — Load both neural predictive heads and verify feature inputs

The original V4 only used binary heads. This stage adds frozen multiclass heads and checks probability shapes. Neural inference is cached to avoid repeated GPU use.

In [ ]:
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon
from pathlib import Path
import hashlib
WP2_OUT=Path(V2)/'WP2_detector_specific'; WP2_OUT.mkdir(parents=True,exist_ok=True)
MULTI_PATHS={
    'MLP':f'{BASE}/models/mlp/MLP_multiclass_seed42.keras',
    'FTTransformer':f'{BASE}/models/ft_transformer/FTTransformer_multiclass_seed42.keras'
}
MM={k:keras.models.load_model(v,compile=False,custom_objects=CUSTOM,safe_mode=False) for k,v in MULTI_PATHS.items()}
for name, model in MM.items():
    assert model.input_shape[-1]==68
    print(name,'multiclass output shape',model.output_shape)

def p_multi(model,X,bs=8192):
    p=np.asarray(model.predict(X.astype('float32'),batch_size=bs,verbose=0),dtype=np.float64)
    assert p.ndim==2 and p.shape[1]==9, ('Expected 9 multiclass outputs',p.shape)
    assert np.isfinite(p).all() and (p>=0).all()
    p=p/np.sum(p,axis=1,keepdims=True)
    return p

def temp_softmax(p,T):
    p=np.clip(np.asarray(p,dtype=np.float64),1e-15,1)
    z=np.log(p)/T; z-=z.max(axis=1,keepdims=True)
    e=np.exp(z); return e/e.sum(axis=1,keepdims=True)

# No multiclass labels are used for temperature calibration in this extension:
# T_multi=1.0 (pre-specified, not fitted on deployment).
T_MULTI={k:1.0 for k in MM}
print('Multiclass temperature policy:',T_MULTI)


## N3 — Frozen-style monitor mathematics

Wasserstein is divided by the source IQR, falling back to source SD, then zero if both vanish. Jensen–Shannon **divergence** is squared JS distance (base 2). Twelve raw indicators are normalized by their source-null median-to-q95 exceedance, then aggregated into three family medians and a final median.

In [ ]:
WINDOW=5000; SEEDS=[1,7,21,42]; EPS=0.01
IND_CONF=['binary_confidence_shift','binary_entropy_shift','binary_margin_shift',
          'multiclass_confidence_shift','multiclass_entropy_shift','multiclass_margin_shift']
IND_CLASS=['binary_prediction_js','multiclass_prediction_js']
IND_FEATURE=['feature_shift_mean','feature_shift_median','feature_shift_p90','feature_shift_max']
ALL_IND=IND_CONF+IND_CLASS+IND_FEATURE

def wd(ref,cur):
    ref=np.asarray(ref,np.float64); cur=np.asarray(cur,np.float64)
    q25,q75=np.quantile(ref,[.25,.75]); scale=q75-q25
    if not np.isfinite(scale) or scale<1e-12: scale=np.std(ref)
    if not np.isfinite(scale) or scale<1e-12:return 0.
    return float(wasserstein_distance(ref,cur)/scale)

def js(ref,cur,nclass):
    a=np.bincount(np.asarray(ref,dtype=np.int64),minlength=nclass).astype(float)+1e-12
    b=np.bincount(np.asarray(cur,dtype=np.int64),minlength=nclass).astype(float)+1e-12
    return float(jensenshannon(a/a.sum(),b/b.sum(),base=2)**2)

def make_signals(pb,pm):
    pb=np.clip(np.asarray(pb,np.float64),1e-15,1-1e-15)
    pm=np.clip(np.asarray(pm,np.float64),1e-15,1)
    pm=pm/pm.sum(axis=1,keepdims=True)
    b=np.column_stack([1-pb,pb]); order=np.sort(pm,axis=1)
    return {'binary_confidence':np.max(b,axis=1),
            'binary_entropy':-np.sum(b*np.log(b),axis=1)/np.log(2),
            'binary_margin':np.abs(2*pb-1),
            'binary_predicted_class':np.argmax(b,axis=1),
            'multiclass_confidence':np.max(pm,axis=1),
            'multiclass_entropy':-np.sum(pm*np.log(pm),axis=1)/np.log(9),
            'multiclass_margin':order[:,-1]-order[:,-2],
            'multiclass_predicted_class':np.argmax(pm,axis=1)}

def feature_indicators(Xref,Xwin):
    d=np.array([wd(Xref[:,j],Xwin[:,j]) for j in range(68)])
    return dict(zip(IND_FEATURE,[float(d.mean()),float(np.median(d)),
                                  float(np.quantile(d,.9)),float(d.max())]))

def predictive_indicators(ref,cur):
    d={c+'_shift':wd(ref[c],cur[c]) for c in
       ['binary_confidence','binary_entropy','binary_margin',
        'multiclass_confidence','multiclass_entropy','multiclass_margin']}
    d['binary_prediction_js']=js(ref['binary_predicted_class'],cur['binary_predicted_class'],2)
    d['multiclass_prediction_js']=js(ref['multiclass_predicted_class'],cur['multiclass_predicted_class'],9)
    return d

def fit_norm(raw):
    out={}
    for c in ALL_IND:
        a=np.asarray(raw[c],float); med=float(np.median(a)); q95=float(np.quantile(a,.95))
        # A zero-null indicator is structural; positive values flag novelty.
        structural=bool(np.all(a==0))
        out[c]={'median':med,'q95':q95,'scale':q95-med,'structural_zero':structural}
    return out

def norm_one(c,v,norm):
    x=norm[c]
    if x['structural_zero']:return float(v>0)
    if x['scale']<=0:return 0.
    return max(0.,(float(v)-x['median'])/x['scale'])

def score(raw,norm):
    E={c:norm_one(c,raw[c],norm) for c in ALL_IND}
    fam={'confidence':float(np.median([E[c] for c in IND_CONF])),
         'predicted_class':float(np.median([E[c] for c in IND_CLASS])),
         'feature':float(np.median([E[c] for c in IND_FEATURE]))}
    return float(np.median(list(fam.values()))),fam,E


## N4 — Source-only monitor calibration (54 windows)

For each detector, 54 disjoint source calibration windows define raw null evidence. Each source-null predictive comparison excludes the target window from its reference. Input features are the **imputed original feature space**, not the neural-scaled feature space. The deployment feature reference is the complete imputed calibration set, matching the original R10 feature definition.

The same source-null feature indicators are reused for both neural detectors. The source-null monitoring boundaries are estimated independently per detector. The binary risk threshold is retained from N1; the multiclass model contributes monitoring evidence, not binary decision labels.

In [ ]:
import gc, json
P_IMP=f'{BASE}/data/processed/CICIDS2017_68F_imputed'
Xcal_raw=pd.read_parquet(f'{P_IMP}/CICIDS2017_calibration_68F_imputed.parquet',columns=FEATURES).to_numpy(np.float64)
assert Xcal_raw.shape==(274787,68)
Xc=pd.read_parquet(f'{P_NEU}/CICIDS2017_calibration_68F_neural.parquet',columns=FEATURES).to_numpy(np.float64)
assert np.max(np.abs(neural_transform(Xcal_raw)-Xc))<1e-9
N_NULL=54; null_indices=np.random.default_rng(42).permutation(len(Xc))[:N_NULL*WINDOW].reshape(N_NULL,WINDOW)
# This uses the exact source-only random permutation from N1.
feature_null=[]
for i,idx in enumerate(null_indices):
    keep=np.ones(len(Xcal_raw),bool);keep[idx]=False
    feature_null.append(feature_indicators(Xcal_raw[keep],Xcal_raw[idx]))
    if (i+1)%9==0:print('source feature null',i+1,'/',N_NULL,flush=True)
feature_null=pd.DataFrame(feature_null)
MON={}
for name in M:
    pb=calibrated(p_attack(M[name],Xc),CAL[name]['T'])
    pm=temp_softmax(p_multi(MM[name],Xc),T_MULTI[name])
    sig=make_signals(pb,pm)
    raw_null=[]
    for i,idx in enumerate(null_indices):
        keep=np.ones(len(Xc),bool);keep[idx]=False
        ref={k:v[keep] for k,v in sig.items()}
        cur={k:v[idx] for k,v in sig.items()}
        raw_null.append({**predictive_indicators(ref,cur),**feature_null.iloc[i].to_dict()})
    raw_null=pd.DataFrame(raw_null)
    norm=fit_norm(raw_null)
    scores=np.array([score(r,norm)[0] for r in raw_null.to_dict('records')])
    boundaries=np.quantile(scores,[.95,.99]).tolist()
    MON[name]={'norm':norm,'boundaries':boundaries,'reference':sig,
               'source_sss':scores,'source_raw':raw_null}
    raw_null.to_csv(WP2_OUT/f'WP2_V9_{name}_source_null_raw.csv',index=False)
    pd.DataFrame({'window':np.arange(N_NULL),'SSS':scores}).to_csv(WP2_OUT/f'WP2_V9_{name}_source_null_sss.csv',index=False)
    with open(WP2_OUT/f'WP2_V9_{name}_frozen_monitor.json','w') as f:
        json.dump({'normalization':norm,'boundaries':boundaries,
                   'binary_temperature':CAL[name]['T'],'multiclass_temperature':T_MULTI[name],
                   'calibration_only':True,'window_size':WINDOW,'n_source_null':N_NULL,
                   'feature_space':'original imputed 68 features',
                   'monitor_families':3,'source_only':True},f,indent=2)
    print(name,'source-only q95/q99',boundaries,'null state counts',
          pd.Series(np.where(scores<=boundaries[0],'normal',np.where(scores<=boundaries[1],'elevated','severe'))).value_counts().to_dict())
    del pb,pm,sig
    gc.collect()
print('N4 source calibration frozen; no deployment labels used.')


## N5 — Deployment monitoring without labels

Evaluate the four fixed 60-window orderings. The 68-dimensional Wasserstein feature indicators are shared between detectors and cached once per window. Results are saved **before** loading deployment labels. The script asserts exact source-row indexing.

In [ ]:
monitor_rows=[]
for seed in SEEDS:
    path=f'{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{seed}_FLOAT64_STREAM_FEATURES.npz'
    with np.load(path,allow_pickle=False) as z:
        assert z['feature_order'].tolist()==FEATURES
        Xraw=np.asarray(z['X'],dtype=np.float64)
        source_idx=np.asarray(z['source_row_index'],dtype=np.int64)
    assert len(Xraw)==300000 and len(source_idx)==len(Xraw)
    assert source_idx.min()>=0 and source_idx.max()<len(X_iid_raw)
    # For auditability, validate the saved raw features against indexed IID rows.
    match=float(np.max(np.abs(Xraw[:100]-X_iid_raw[source_idx[:100]])))
    assert match<1e-8, f'Stream index/feature mismatch {match}'
    Xneu=neural_transform(Xraw)
    feat=[]
    for w in range(60):
        sl=slice(w*WINDOW,(w+1)*WINDOW)
        feat.append(feature_indicators(Xcal_raw,Xraw[sl]))
    for name in M:
        pb=calibrated(p_attack(M[name],Xneu),CAL[name]['T'])
        pm=temp_softmax(p_multi(MM[name],Xneu),T_MULTI[name])
        sig=make_signals(pb,pm)
        spec=MON[name]
        for w in range(60):
            sl=slice(w*WINDOW,(w+1)*WINDOW)
            cur={k:v[sl] for k,v in sig.items()}
            raw={**predictive_indicators(spec['reference'],cur),**feat[w]}
            s,fam,E=score(raw,spec['norm'])
            lo,hi=spec['boundaries']
            state='normal' if s<=lo else ('elevated' if s<=hi else 'severe')
            monitor_rows.append({'seed':seed,'window':w,'detector':name,'SSS':s,'state':state,
                                 **{'family_'+k:v for k,v in fam.items()},
                                 **{'raw_'+k:v for k,v in raw.items()},
                                 **{'E_'+k:v for k,v in E.items()}})
        del pb,pm,sig
        gc.collect()
    print('monitoring complete seed',seed,flush=True)
    del Xraw,Xneu
    gc.collect()
monitor_df=pd.DataFrame(monitor_rows)
assert len(monitor_df)==480
assert monitor_df.groupby(['seed','detector']).size().eq(60).all()
monitor_df.to_csv(WP2_OUT/'WP2_V9_label_free_monitor_windows.csv',index=False)
print(monitor_df.groupby(['detector','state']).size())
print('LABEL-FREE MONITORING SAVED; no deployment outcomes evaluated.')


## N6 — Outcome evaluation, frozen thresholds, and policy comparison

Only now load deployment binary labels. Policies: no abstention, fixed Normal, static Severe, and detector-specific three-family. **Transferred XGBoost states are not fabricated**; they can be evaluated separately if exact original streaming-state files are located. A 1% *empirical* selective-risk violation is counted per window. Abstention has no formal risk guarantee.

In [ ]:
def count_window(y,p,t):
    pred=(p>=.5); risk=1-np.maximum(p,1-p)
    accept=risk<=t
    attack=(y==1); err=(pred!=y)
    return dict(n=len(y),acc_n=int(accept.sum()),acc_err=int((accept&err).sum()),
                n_attack=int(attack.sum()),deferred_attack=int((~accept&attack).sum()),
                acc_detected_attack=int((accept&attack&pred).sum()),
                acc_missed_attack=int((accept&attack&~pred).sum()),
                acc_false_alarm=int((accept&~attack&pred).sum()))

rows=[]
for seed in SEEDS:
    path=f'{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{seed}_FLOAT64_STREAM_FEATURES.npz'
    with np.load(path,allow_pickle=False) as z:
        Xraw=np.asarray(z['X'],np.float64)
        idx=np.asarray(z['source_row_index'],np.int64)
    y=y_iid[idx]
    Xneu=neural_transform(Xraw)
    for name in M:
        pb=calibrated(p_attack(M[name],Xneu),CAL[name]['T'])
        th=CAL[name]['thresholds']
        states=monitor_df[(monitor_df.seed==seed)&(monitor_df.detector==name)].sort_values('window')
        assert states.window.tolist()==list(range(60))
        for w,state in enumerate(states.state):
            sl=slice(w*WINDOW,(w+1)*WINDOW)
            choices={'no_abstention':0.5,'fixed_normal':th['fixed'],
                     'static_severe':th['severe'],
                     'detector_specific_3family':th[{'normal':'fixed','elevated':'elevated','severe':'severe'}[state]]}
            for policy,t in choices.items():
                rows.append(dict(seed=seed,window=w,detector=name,policy=policy,
                                 state=state,threshold=float(t),**count_window(y[sl],pb[sl],t)))
        del pb
    del Xraw,Xneu
    gc.collect()
    print('outcomes complete seed',seed,flush=True)
res=pd.DataFrame(rows)
assert len(res)==1920 and res.groupby(['seed','detector','policy']).size().eq(60).all()
res.to_csv(WP2_OUT/'WP2_V9_window_policy_results.csv',index=False)

def summarize(g):
    risk=np.where(g.acc_n>0,g.acc_err/g.acc_n,np.nan)
    compliant=np.isfinite(risk)&(risk<=EPS)
    return pd.Series({'windows':len(g),'violations':int(np.nansum(risk>EPS)),
       'coverage':g.acc_n.sum()/g.n.sum(),
       'accepted_risk':g.acc_err.sum()/max(g.acc_n.sum(),1),
       'attack_deferral':g.deferred_attack.sum()/max(g.n_attack.sum(),1),
       'auto_attack_recall':g.acc_detected_attack.sum()/max(g.n_attack.sum(),1),
       'bcdr':g.loc[compliant,'acc_detected_attack'].sum()/max(g.n_attack.sum(),1),
       'false_alarms':int(g.acc_false_alarm.sum())})
summary=res.groupby(['detector','policy']).apply(summarize,include_groups=False).reset_index()
summary.to_csv(WP2_OUT/'WP2_V9_policy_summary.csv',index=False)
display(summary)
print('V9 OUTCOME EVALUATION COMPLETE. Transferred-XGBoost-state baseline: NOT INCLUDED (requires verified original state alignment).')


## N7 — Audit manifest and reporting gates

Report all model-specific states, risk/coverage, and missing comparisons. **Do not claim detector-specific monitoring outperforms transferred XGBoost states** until the transferred-state baseline is implemented with matching seed, window, and source indices. No threshold tuning or statistical significance claims should be made from these deployment results.

In [ ]:
import json
manifest={'version':'WP2_V9','detectors':list(M),'seeds':SEEDS,'windows_per_seed':60,
          'window_size':WINDOW,'n_monitor_rows':len(monitor_df),'n_policy_rows':len(res),
          'source_only_monitor_calibration':True,'deployment_labels_used_for_monitor':False,
          'historical_xgboost_replay':'V8 PASS (separate notebook)',
          'transferred_xgboost_baseline':'not evaluated',
          'multiclass_temperature':'1.0 fixed (not calibrated)',
          'model_retraining':False,
          'limitations':['source-null calibration uses 54 windows',
                         'monitor family structure matches original but neural source normalization is newly estimated',
                         'no prospective comparison to transferred XGBoost states yet',
                         'empirical 1% risk threshold does not guarantee target risk']}
with open(WP2_OUT/'WP2_V9_manifest.json','w') as f:json.dump(manifest,f,indent=2)
print('Saved artifacts:')
for p in sorted(WP2_OUT.glob('WP2_V9_*')):print(p.name)
print('NEXT: send N5 state counts and N6 policy summary. V10 will add verified transferred states and statistical analysis.')
